# Consolidate InCites exports and check ISSN coverage

1. Reads every InCites *Publication Sources* workbook in a folder and stacks them into **one CSV**.
2. Compares the result against your ISSN list and sorts each journal into one of three groups:
   * **found**: at least one of the journal's ISSNs is in the InCites data
   * **searched, not found**: the ISSN was in an InCites search (from each workbook's *Result Summary* filter list), but no data came back
   * **not searched yet**: the ISSN isn't in any workbook's search filter, so a batch may have been missed

A journal counts as found if **either** its print ISSN or its eISSN is found. That way a journal InCites indexes under its other ISSN isn't listed as missing.

## Settings

In [1]:
import re
from pathlib import Path
import pandas as pd

INCITES_FOLDER = Path("data/Incites")                         # folder of InCites .xlsx exports
ISSN_LIST_FILE = Path("output/02_journal_issns_for_jcr.csv")  # .csv/.xlsx (any column with "issn" in its name) or .txt (one ISSN per line)
OUT_DIR        = Path("output")

DROP_EMPTY_ROWS = True   # InCites pads each journal with a row for every year 1980–2026; drop years with no documents and no JIF

## 1. Consolidate the InCites workbooks

In [2]:
ISSN_RE = re.compile(r"\b(\d{4})-?(\d{3}[\dX])\b", re.I)

def norm_issn(x):
    if pd.isna(x):
        return None
    m = ISSN_RE.search(str(x).upper())
    return f"{m.group(1)}-{m.group(2)}" if m else None

def read_incites_workbook(path):
    """Returns (data, ISSNs listed in the Result Summary search filter)."""
    xl = pd.ExcelFile(path)
    data, searched = None, set()
    for sheet in xl.sheet_names:
        df = pd.read_excel(xl, sheet_name=sheet, dtype=str)
        cols = {c.strip().lower() for c in df.columns.astype(str)}
        if data is None and "name" in cols and ("issn" in cols or "eissn" in cols):
            data = df
        else:
            text = " ".join(pd.read_excel(xl, sheet_name=sheet, header=None, dtype=str)
                              .fillna("").astype(str).values.ravel())
            for line in re.findall(r"ISSN / eISSN[^:]*:\s*\[([^\]]*)\]", text):
                searched.update(f"{a}-{b}".upper() for a, b in ISSN_RE.findall(line))
    return data, searched

files = sorted(p for p in INCITES_FOLDER.glob("*.xls*") if not p.name.startswith("~$"))
if not files:
    raise FileNotFoundError(f"No Excel files in {INCITES_FOLDER.resolve()}")

frames, searched_issns, skipped = [], set(), []
for f in files:
    data, searched = read_incites_workbook(f)
    searched_issns |= searched
    if data is None:
        skipped.append(f.name)
        continue
    frames.append(data.assign(source_file=f.name))
    print(f"{f.name}: {len(data):,} rows, {data['Name'].nunique():,} journals, {len(searched):,} ISSNs in search filter")
if skipped:
    print("\nSkipped (no InCites data sheet found):", skipped)

incites = pd.concat(frames, ignore_index=True)
incites.columns = [c.strip() for c in incites.columns]

# numeric clean-up (Excel float noise like 22.700000000000003)
for c in ["Web of Science Documents", "Times Cited", "Journal Impact Factor", "5 Year Impact Factor",
          "Impact Factor w/o Self Cites", "Journal Citation Indicator", "Average JIF Percentile", "Citation Impact"]:
    if c in incites.columns:
        incites[c] = pd.to_numeric(incites[c], errors="coerce")
for c in ["Journal Impact Factor", "5 Year Impact Factor", "Impact Factor w/o Self Cites", "Journal Citation Indicator"]:
    if c in incites.columns:
        incites[c] = incites[c].round(3)
for c in ["Publication Year", "Web of Science Documents", "Times Cited"]:
    if c in incites.columns:
        incites[c] = pd.to_numeric(incites[c], errors="coerce").round().astype("Int64")
incites["ISSN"]  = incites["ISSN"].map(norm_issn)
incites["eISSN"] = incites["eISSN"].map(norm_issn)

n_raw = len(incites)
if DROP_EMPTY_ROWS:
    incites = incites[incites["Web of Science Documents"].notna() | incites["Journal Impact Factor"].notna()]
n_nonempty = len(incites)
# the same journal-year can appear in more than one export (e.g. a re-search)
incites = incites.drop_duplicates(subset=["Name", "Publication Year", "ISSN", "eISSN"], keep="first")

incites = incites.sort_values(["Name", "Publication Year"]).reset_index(drop=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
incites.to_csv(OUT_DIR / "incites_consolidated.csv", index=False)

print(f"\n{len(files)} workbooks → {n_raw:,} rows"
      + (f" → {n_nonempty:,} with data" if DROP_EMPTY_ROWS else "")
      + f" → {len(incites):,} after removing duplicate journal-years")
print(f"{incites['Name'].nunique():,} journals, {incites['Journal Impact Factor'].notna().sum():,} rows with a JIF")
print("Saved output/incites_consolidated.csv")

Incites Publication Sources_10.xlsx: 2,726 rows, 58 journals, 58 ISSNs in search filter
Incites Publication Sources_11.xlsx: 2,679 rows, 57 journals, 58 ISSNs in search filter
Incites Publication Sources_12.xlsx: 3,243 rows, 69 journals, 68 ISSNs in search filter
Incites Publication Sources_13.xlsx: 3,619 rows, 77 journals, 78 ISSNs in search filter
Incites Publication Sources_14.xlsx: 3,008 rows, 64 journals, 80 ISSNs in search filter
Incites Publication Sources_15.xlsx: 3,525 rows, 75 journals, 80 ISSNs in search filter
Incites Publication Sources_16.xlsx: 3,196 rows, 68 journals, 70 ISSNs in search filter
Incites Publication Sources_17.xlsx: 2,726 rows, 58 journals, 76 ISSNs in search filter
Incites Publication Sources_18.xlsx: 3,196 rows, 68 journals, 77 ISSNs in search filter
Incites Publication Sources_19.xlsx: 2,538 rows, 54 journals, 79 ISSNs in search filter
Incites Publication Sources_1b.xlsx: 1,504 rows, 32 journals, 30 ISSNs in search filter
Incites Publication Sources_2.xl

## 2. Compare against your ISSN list

In [3]:
def load_issn_list(path):
    """One row per journal, with all of its ISSNs."""
    path = Path(path)
    if path.suffix.lower() == ".txt":
        rows = [{"issns": [i]} for i in map(norm_issn, path.read_text().splitlines()) if i]
        return pd.DataFrame(rows).assign(journal=None)
    df = pd.read_excel(path, dtype=str) if path.suffix.lower().startswith(".xls") else pd.read_csv(path, dtype=str)
    issn_cols = [c for c in df.columns if "issn" in c.lower()]
    if not issn_cols:
        raise ValueError(f"No column with 'issn' in its name. Columns: {list(df.columns)}")
    name_col = next((c for c in df.columns if c.lower() in ("journal", "name", "title", "source title")), None)
    out = pd.DataFrame({
        "journal": df[name_col] if name_col else None,
        "issns": df[issn_cols].apply(lambda r: sorted({i for i in map(norm_issn, r) if i}), axis=1),
    })
    return out[out["issns"].str.len() > 0].reset_index(drop=True)

wanted = load_issn_list(ISSN_LIST_FILE)
found_issns = set(incites["ISSN"].dropna()) | set(incites["eISSN"].dropna())

def status(issns):
    if any(i in found_issns for i in issns):
        return "found"
    if any(i in searched_issns for i in issns):
        return "searched, not found"
    return "not searched yet"

wanted["status"] = wanted["issns"].map(status)
wanted["issns_found"] = wanted["issns"].map(lambda l: "; ".join(i for i in l if i in found_issns))
wanted["issns"] = wanted["issns"].str.join("; ")

# Name InCites uses for each found journal (handy for spotting title mismatches)
name_by_issn = {}
for r in incites[["Name", "ISSN", "eISSN"]].drop_duplicates().itertuples(index=False):
    for i in (r.ISSN, r.eISSN):
        if i:
            name_by_issn.setdefault(i, r.Name)
wanted["incites_name"] = wanted["issns_found"].map(lambda s: name_by_issn.get(s.split("; ")[0]) if s else None)

wanted.to_csv(OUT_DIR / "issn_coverage_in_incites.csv", index=False)
missing = wanted[wanted["status"] != "found"]
missing.to_csv(OUT_DIR / "issns_not_found_in_incites.csv", index=False)

to_research = sorted({i for s in missing["issns"] for i in s.split("; ")})
(OUT_DIR / "issns_to_research.txt").write_text("\n".join(to_research))

print(wanted["status"].value_counts().to_string())
print(f"\n{len(to_research)} ISSNs to search again → output/issns_to_research.txt")
print("Comma-separated, for pasting into the InCites ISSN filter:\n")
print(", ".join(to_research))

status
found                  1104
not searched yet         57
searched, not found       2

83 ISSNs to search again → output/issns_to_research.txt
Comma-separated, for pasting into the InCites ISSN filter:

0069-4827, 0098-6577, 0146-6453, 0277-786X, 0361-7742, 0907-4449, 0926-9630, 0927-3042, 0973-1040, 1052-6773, 1064-3745, 1080-2371, 1175-2203, 1179-7274, 1345-711X, 1357-714X, 1369-1643, 1399-0047, 1538-6899, 1539-3755, 1543-1894, 1548-8748, 1548-8756, 1550-2376, 1570-0267, 1574-8898, 1745-6614, 1872-969X, 1879-8365, 1929-6029, 1933-9631, 1933-964X, 1934-2500, 1934-2616, 1934-3396, 1934-340X, 1940-6029, 1945-7928, 1945-8452, 1996-756X, 2041-6792, 2041-6806, 2045-0907, 2045-0915, 2045-1393, 2045-1407, 2046-1402, 2049-9256, 2152-7806, 2157-7633, 2159-2780, 2159-2799, 2167-0870, 2229-5097, 2254-6081, 2329-8901, 2331-091X, 2331-0928, 2331-4737, 2331-8422, 2350-269X, 2374-684X, 2375-1916, 2375-1924, 2375-7477, 2468-4988, 2534-773X, 2574-7754, 2613-4942, 2666-2469, 2666-853X, 2666-8548, 

## 3. Not found: details

In [4]:
missing.sort_values(["status", "journal"], na_position="last")

,journal,issns,status,issns_found,incites_name
944,Advanced health care technologies,2350-269X,not searched yet,,None
536,Advances in oncology,2666-853X; 2666-8548,not searched yet,,None
513,American Society of Clinical Oncology educatio...,1548-8748; 1548-8756,not searched yet,,None
659,American journal of pharmacogenomics : genomic...,1175-2203,not searched yet,,None
1116,Annals of case reports,2574-7754,not searched yet,,None
1010,Annals of the ICRP,0146-6453; 1872-969X,not searched yet,,None
985,Annual International Conference of the IEEE En...,2375-7477; 2694-0604,not searched yet,,None
169,ArXiv,2331-8422,not searched yet,,None
923,Archives in cancer research,2254-6081,not searched yet,,None
316,Blood neoplasia,2950-3280,not searched yet,,None


## Output files
| File | Contents |
|---|---|
| `incites_consolidated.csv` | All InCites rows from every workbook, with a `source_file` column |
| `issn_coverage_in_incites.csv` | Every journal on your list with its status |
| `issns_not_found_in_incites.csv` | Only the journals not found |
| `issns_to_research.txt` | ISSNs to search in InCites again, one per line |

Once you've re-searched, save the new export(s) in the same folder and re-run the notebook. Duplicate journal-years are removed automatically.